# Exercise 21.1: Token Embeddings & Positional Assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-21/01-token-embeddings-and-positional-assembly.ipynb)

- **Book Reference:** Chapter 21 (*Mathematics of Transformers and Large Language Models*, Section 21.1)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Map discrete token IDs through a vocabulary embedding table $E \in \mathbb{R}^{V \times d}$ to form $X \in \mathbb{R}^{T \times d}$.
  2. Construct sinusoidal positional encodings $\mathrm{PE}(t, 2i) = \sin(t / 10000^{2i/d})$ and $\mathrm{PE}(t, 2i+1) = \cos(t / 10000^{2i/d})$.
  3. Assemble the Transformer input $X + \mathrm{PE}$ and verify that identical tokens at different positions receive distinct representations.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## Mathematical Background

A Transformer converts a sequence of $T$ token IDs $(w_1, \dots, w_T)$ into a matrix $X \in \mathbb{R}^{T \times d}$ by looking up rows of an embedding matrix $E \in \mathbb{R}^{V \times d}$.

Because self-attention is permutation-equivariant, we add a **sinusoidal positional encoding** $\mathrm{PE} \in \mathbb{R}^{T \times d}$:
$$\mathrm{PE}[t, 2i] = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad \mathrm{PE}[t, 2i+1] = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$
for position $t \in \{0, \dots, T-1\}$ and dimension index $i \in \{0, \dots, d/2 - 1\}$.

In [ ]:
def embed_tokens(token_ids: np.ndarray, embedding_table: np.ndarray) -> np.ndarray:
    """Look up token embeddings for a sequence of token IDs.

    Args:
        token_ids: 1D integer array of shape (T,).
        embedding_table: Vocabulary matrix of shape (V, d_model).

    Returns:
        Token embedding matrix X of shape (T, d_model).
    """
    # TODO: Index embedding_table with token_ids
    raise NotImplementedError("Complete this TODO")


def sinusoidal_positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """Generate Transformer sinusoidal positional encoding matrix of shape (seq_len, d_model).

    Args:
        seq_len: Sequence length T.
        d_model: Embedding dimension d (must be even).

    Returns:
        Positional encoding matrix PE of shape (seq_len, d_model).
    """
    # TODO: Compute sin on even indices 2i and cos on odd indices 2i+1
    raise NotImplementedError("Complete this TODO")


def assemble_transformer_input(
    token_ids: np.ndarray, embedding_table: np.ndarray
) -> np.ndarray:
    """Compute X + PE for a token ID sequence."""
    # TODO: Combine embed_tokens and sinusoidal_positional_encoding
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- `embedding_table[token_ids]` performs vectorised row lookup in NumPy.
- Use broadcasting: `positions = np.arange(seq_len)[:, None]` of shape `(T, 1)` and `10000 ** (-np.arange(0, d_model, 2) / d_model)` of shape `(d_model // 2,)`.
- Fill `pe[:, 0::2]` with `np.sin(...)` and `pe[:, 1::2]` with `np.cos(...)`.

</details>

In [ ]:
V, d_model = 50, 16
E = rng.normal(0.0, 0.5, size=(V, d_model))

# Sequence with repeated token ID 7 at positions 0 and 3
tokens = np.array([7, 12, 4, 7, 19])
PE = sinusoidal_positional_encoding(len(tokens), d_model)
H0 = assemble_transformer_input(tokens, E)

assert PE.shape == (5, 16)
assert H0.shape == (5, 16)
# At position 0: sin(0)=0, cos(0)=1
assert np.allclose(PE[0, 0::2], 0.0)
assert np.allclose(PE[0, 1::2], 1.0)
# Every sin^2 + cos^2 pair sums to 1, so ||PE[t]||^2 == d_model / 2
assert np.allclose(np.sum(PE ** 2, axis=1), d_model / 2.0)
# Repeated token 7 at pos 0 vs pos 3 now has distinct vectors!
assert not np.allclose(H0[0], H0[3])
print("All checks passed! Assembled shape:", H0.shape)

## Think Deeper

1. Why do Transformers add positional encodings to token embeddings instead of leaving $X$ as-is (Exercise 21.1)?
2. Why does the sinusoidal encoding use exponentially spaced wavelengths from $2\pi$ to $10000 \cdot 2\pi$?